# Research-Only BRCA1/BRCA2 Variant Annotation Case Study

This notebook creates reproducible Ensembl VEP annotations for four GRCh38 variants curated from the public Kaggle source described in the repository README. It is an educational portfolio exercise, **not** a clinical laboratory pipeline or diagnostic report.

**Before running:** upload `data/input_variants.csv` to the Colab working directory or clone the GitHub repository into the runtime.


In [ ]:
# Optional: clone your own GitHub repository in Google Colab.
# Replace the URL after you create the repository, then uncomment the line below.
# !git clone https://github.com/YOUR-USERNAME/research-variant-annotation-case-study.git

from pathlib import Path
from datetime import date
import json
import pandas as pd
import requests

# Use the cloned repository path if applicable; otherwise upload input_variants.csv and set DATA_PATH accordingly.
DATA_PATH = Path('data/input_variants.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Upload data/input_variants.csv or update DATA_PATH to its Colab location.')

results_dir = Path('results')
results_dir.mkdir(exist_ok=True)
variants = pd.read_csv(DATA_PATH)
variants[['variant_id', 'gene', 'assembly', 'chromosome', 'position', 'ref', 'alt', 'hgvs_c_source']]


## Step 1 — Confirm input provenance and genome build

All supplied coordinates are GRCh38. Keep the original VCF representation even if VEP normalises an indel internally. The original VCF representation and the VEP-normalised result are both useful for a transparent audit trail.


In [ ]:
assert variants['assembly'].eq('GRCh38').all(), 'Do not mix genome builds in this small project.'

def vep_region_string(row):
    rsid = row['dbsnp_rsid'] if pd.notna(row['dbsnp_rsid']) else '.'
    return f"{row['chromosome']} {row['position']} {rsid} {row['ref']} {row['alt']} . . ."

vep_inputs = variants.apply(vep_region_string, axis=1).tolist()
print('VEP inputs:')
print('\n'.join(vep_inputs))


## Step 2 — Request Ensembl VEP annotations

Ensembl VEP predicts molecular consequences for variants. This is an annotation step, not a clinical classification. The request uses the documented public POST endpoint and saves the unmodified response for reproducibility.[1]

[1]: https://rest.ensembl.org/documentation/info/vep_region_post "Ensembl REST: POST VEP region


In [ ]:
endpoint = 'https://rest.ensembl.org/vep/homo_sapiens/region'
headers = {'Content-Type': 'application/json', 'Accept': 'application/json'}
response = requests.post(endpoint, headers=headers, json={'variants': vep_inputs}, timeout=120)
response.raise_for_status()
vep_records = response.json()

stamp = date.today().isoformat()
raw_output = results_dir / f'vep_annotation_{stamp}.json'
raw_output.write_text(json.dumps(vep_records, indent=2), encoding='utf-8')
print(f'Saved {len(vep_records)} VEP records to {raw_output}')


## Step 3 — Create a compact annotation table

This cell extracts one transcript consequence per VEP record for a readable portfolio table. Transcript choice is a substantive issue in clinical practice; the code therefore retains the full raw VEP response and labels this output as a convenience summary only.


In [ ]:
def choose_transcript(record):
    consequences = record.get('transcript_consequences', [])
    if not consequences:
        return {}
    return next((c for c in consequences if c.get('canonical') == 1), consequences[0])

def frequency_context(record):
    # VEP may normalise a deletion; find the frequency dictionary for the ALT allele in the output allele string.
    output_alt = record.get('allele_string', '').split('/')[-1]
    for colocated in record.get('colocated_variants', []):
        values = colocated.get('frequencies', {}).get(output_alt)
        if values:
            return {
                'colocated_id': colocated.get('id'),
                'gnomade': values.get('gnomade'),
                'gnomadg': values.get('gnomadg'),
            }
    return {'colocated_id': None, 'gnomade': None, 'gnomadg': None}

rows = []
for source_row, record in zip(variants.to_dict('records'), vep_records):
    transcript = choose_transcript(record)
    frequency = frequency_context(record)
    rows.append({
        'variant_id': source_row['variant_id'],
        'gene_source': source_row['gene'],
        'input_vcf': record.get('input'),
        'vep_allele_string': record.get('allele_string'),
        'most_severe_consequence': record.get('most_severe_consequence'),
        'transcript_id': transcript.get('transcript_id'),
        'gene_vep': transcript.get('gene_symbol'),
        'consequence_terms': ';'.join(transcript.get('consequence_terms', [])),
        'impact': transcript.get('impact'),
        'hgvsc_vep': transcript.get('hgvsc'),
        'hgvsp_vep': transcript.get('hgvsp'),
        **frequency,
    })

vep_table = pd.DataFrame(rows)
flat_output = results_dir / f'vep_flattened_{stamp}.csv'
vep_table.to_csv(flat_output, index=False)
vep_table


## Step 4 — Record public evidence with appropriate limits

Use the source identifiers in `input_variants.csv` to check the live ClinVar record, gnomAD record, and a cancer-specific resource. Copy `results/research_evidence_summary_template.csv` to `results/research_evidence_summary.csv`, complete it manually, and record the date and stable URL/accession for each review.

- **ClinVar:** record the published assertion and review status; do not call it your own classification.[1]
- **gnomAD:** record build/dataset and a frequency observation; do not apply a threshold mechanically.[2]
- **CIViC/cBioPortal:** identify whether the result is exact-variant evidence, gene-level context, or no exact record found on the review date.[3] [4]

[1]: https://www.ncbi.nlm.nih.gov/clinvar/ "ClinVar"
[2]: https://gnomad.broadinstitute.org/help "gnomAD"
[3]: https://docs.civicdb.org/ "CIViC"
[4]: https://docs.cbioportal.org/web-api-and-clients/ "cBioPortal


In [ ]:
evidence_template = pd.read_csv('results/research_evidence_summary_template.csv')
evidence_template[['variant_id', 'source_clinvar_assertion', 'source_review_status', 'research_only_interpretation']]


## Required final disclaimer

> **Research-only interpretation.** This portfolio exercise summarises publicly available, database-derived evidence for educational and research purposes. It is not a validated clinical laboratory analysis; it does not establish a diagnosis, a patient-specific risk assessment, a clinical variant classification, or a treatment recommendation. Published ClinVar assertions are reported as source data and are not independently reclassified here.
